In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/nq_first_to_100_predictions_evaluated.csv")
df = df.dropna(how="all").reset_index(drop=True)

In [3]:
df.head()

,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2024-09-02,Monday,Long,61%,Moderate Up,Trend Continuation,Invalid,NaN,Positive ORG followed an overnight sell-side s...
1,2024-09-03,Tuesday,Short,63%,Strong Down,Trend Continuation,Short,True,Large negative ORG accompanied a persistent de...
2,2024-09-04,Wednesday,Short,66%,Strong Down,Trend Continuation,Long,False,Large negative ORG followed an exceptionally w...
3,2024-09-05,Thursday,Short,59%,Moderate Down,Balance,Long,False,Negative ORG developed within a choppy overnig...
4,2024-09-06,Friday,Long,62%,Moderate Up,Exhaustion,Short,False,Nearly flat ORG followed a broad overnight ran...


In [4]:
df["Confidence Numeric"] = (
    df["Confidence"]
    .str.rstrip("%")
    .astype(int)
)

In [5]:
confidence_accuracy = (
    df.dropna(subset=["Correct"])
      .groupby("Confidence Numeric")["Correct"]
      .agg(["count", "mean"])
)

confidence_accuracy["Accuracy %"] = confidence_accuracy["mean"] * 100

confidence_accuracy

,count,mean,Accuracy %
Confidence Numeric,,,
53,1,0.0,0.0
54,2,0.5,50.0
55,4,0.25,25.0
56,12,0.583333,58.333333
57,19,0.526316,52.631579
58,48,0.5625,56.25
59,39,0.512821,51.282051
60,30,0.6,60.0
61,43,0.674419,67.44186


In [6]:
valid_df = df.dropna(subset=["Correct"]).copy()

valid_df["Confidence Group"] = pd.cut(
    valid_df["Confidence Numeric"],
    bins=[0, 58, 62, 65, 100],
    labels=["≤58", "59–62", "63–65", "≥66"]
)

In [7]:
confidence_groups = (
    valid_df.groupby("Confidence Group", observed=True)["Correct"]
    .agg(["count", "sum", "mean"])
)

confidence_groups["Accuracy %"] = confidence_groups["mean"] * 100

confidence_groups

,count,sum,mean,Accuracy %
Confidence Group,,,,
≤58,86,46,0.534884,53.488372
59–62,146,87,0.59589,59.589041
63–65,106,55,0.518868,51.886792
≥66,144,69,0.479167,47.916667


In [8]:
valid_df["Confidence Half"] = valid_df["Confidence Numeric"].apply(
    lambda x: "≤62" if x <= 62 else ">62"
)

confidence_halves = (
    valid_df.groupby("Confidence Half")["Correct"]
    .agg(["count", "sum", "mean"])
)

confidence_halves["Accuracy %"] = confidence_halves["mean"] * 100

confidence_halves

,count,sum,mean,Accuracy %
Confidence Half,,,,
>62,250,124,0.496,49.6
≤62,232,133,0.573276,57.327586


In [9]:
overall_accuracy = valid_df["Correct"].mean() * 100

print(f"Overall accuracy: {overall_accuracy:.2f}%")
print(f"Correct: {valid_df['Correct'].sum()} / {len(valid_df)}")

Overall accuracy: 53.32%
Correct: 257 / 482
